# Chapter 7 — How neural networks compute · Annotated solution

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch07_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each answer starts with the reasoning on paper, then shows its equivalent in Python.

**Contents.** Data · The computations of lessons 7.1 to 7.3 · Lesson 7.4 (mini-batches and epochs) · Optional workshop (twelve messages) · Exercises 7.1 to 7.6

**Before opening Jupyter:** first do by hand the forward pass of M7 (lesson 7.1) and the derivative with respect to $w_{11}$ (lesson 7.2). The cells below are there to **check** these computations.

**Color code:** indigo = forward computation; dashed magenta = path of the gradient; thick amber line = studied weight $w_{11}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The book's color code: forward computation in indigo, gradient in magenta (dashed), studied weight in thick amber
FORWARD, BACKWARD, STUDIED_WEIGHT = '#3949ab', '#c2185b', '#f9a825'


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def loss(y, p):
    # log loss of one message (equation 5.6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)))


def forward(P, x):
    # Forward pass (equations 7.1 and 7.2): returns a dictionary z1, a1, z2, a2, z, p
    z1 = P['w11'] * x[0] + P['w12'] * x[1] + P['b1']
    z2 = P['w21'] * x[0] + P['w22'] * x[1] + P['b2']
    a1, a2 = sigmoid(z1), sigmoid(z2)
    z = P['v1'] * a1 + P['v2'] * a2 + P['c']
    return {'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z': z, 'p': sigmoid(z)}


def draw(P, x, backward=False):
    # The book's diagram, simplified: indigo for the forward pass, magenta for the backward pass, w11 in thick amber
    v = forward(P, x)
    pos = {'x1': (0, 3), 'x2': (0, 0), 'n1': (3, 3), 'n2': (3, 0), 's': (6, 1.5)}
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for a, b, name in (('x1', 'n1', 'w11'), ('x2', 'n1', 'w12'), ('x1', 'n2', 'w21'), ('x2', 'n2', 'w22'),
                      ('n1', 's', 'v1'), ('n2', 's', 'v2')):
        (xa, ya), (xb, yb) = pos[a], pos[b]
        if name == 'w11':
            ax.plot([xa, xb], [ya, yb], color=STUDIED_WEIGHT, linewidth=7, zorder=1)
        ax.annotate('', xy=(xb - 0.35, yb + (ya - yb) * 0.12), xytext=(xa + 0.3, ya),
                    arrowprops=dict(arrowstyle='-|>', color=FORWARD), zorder=2)
        t = 0.3 if a[0] == 'x' and b != ('n1' if a == 'x1' else 'n2') else 0.5
        ax.text(xa + t * (xb - xa), ya + t * (yb - ya) + 0.15, f"{name} = {P[name]:g}" + (' (studied weight)' if name == 'w11' else ''),
                color=FORWARD, fontsize=8, ha='center', bbox=dict(facecolor='white', edgecolor='none', alpha=0.85))
    texts = {'x1': f"x1 = {x[0]}", 'x2': f"x2 = {x[1]}",
              'n1': f"N1\nz1 = {v['z1']:.3f}\na1 = {v['a1']:.4f}", 'n2': f"N2\nz2 = {v['z2']:.3f}\na2 = {v['a2']:.4f}",
              's': f"output\nz = {v['z']:.3f}\np̂ = {v['p']:.3f}"}
    for k, (cx, cy) in pos.items():
        ax.text(cx, cy, texts[k], ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='circle', facecolor='white', edgecolor=FORWARD if k[0] != 'x' else 'gray'))
    if backward:
        for (x0, y0), (x1, y1), lab in (((6.6, 2.4), (3.5, 3.6), '② v1'), ((3.0, 3.9), (1.4, 3.4), '③ a1(1 − a1), ④ x1')):
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='-|>', color=BACKWARD, linestyle='--', connectionstyle='arc3,rad=0.3'))
            ax.text((x0 + x1) / 2, max(y0, y1) + 0.35, lab, color=BACKWARD, fontsize=8, ha='center')
        ax.text(6.6, 2.7, '① p̂ − y', color=BACKWARD, fontsize=8)
    ax.set_xlim(-0.8, 7.4); ax.set_ylim(-0.8, 4.4); ax.axis('off'); plt.show()

## Data

The identifiers D7.1, D7.E1 and D5.1 are those of the book and of the CSV files.

In [ ]:
# Chapter 7 data, copied from the book's CSV files (do not modify)

# D7.1 — the nine parameters of the network of lessons 7.1 to 7.3
network = {'w11': 0.5, 'w12': 0.5, 'b1': -2, 'w21': 1, 'w22': -1, 'b2': 0, 'v1': 2, 'v2': -2, 'c': 0}

# D7.E1 — the colleague's network (exercises 7.1 and 7.2)
colleague = {'w11': 1, 'w12': 1, 'b1': -4, 'w21': -1, 'w22': 2, 'b2': -1, 'v1': 2, 'v2': 1, 'c': -2}

# D5.1 — the twelve messages of chapter 5: label, links, suspicious words
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

M7 = (2, 3)   # message M7: 2 links, 3 suspicious words, spam (y = 1)
M3 = (4, 0)   # newsletter M3: 4 links, 0 suspicious words, legitimate (y = 0)

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `message` | message identifier (M1…) | — | — |
| `label` | true label: spam or legitimate | — | `etiquette` |
| `links` | number of links in the message | links | `liens` |
| `suspicious_words` | number of suspicious words in the message | words | `mots_suspects` |
| `network` | the nine parameters of the network of lesson 7.1 (weights, biases, constant) | — | — |
| `colleague` | the nine parameters of the colleague's network | — | — |
| `twelve` | the twelve messages of chapter 5 | — | — |

## The computations of the lessons, in Python

These cells redo the computations of lessons 7.1 to 7.3.

In [ ]:
# Lesson 7.1 — forward pass of message M7 (equations 7.1 and 7.2)
v = forward(network, M7)
print(f"N1: z1 = {v['z1']:.3f}, a1 = {v['a1']:.4f}")
print(f"N2: z2 = {v['z2']:.3f}, a2 = {v['a2']:.4f}")
print(f"output: z = {v['z']:.3f}, p̂ = {v['p']:.3f}, loss = {loss(1, v['p']):.3f}")
print(f"M3 (newsletter): a2 = {forward(network, M3)['a2']:.3f}, p̂ = {forward(network, M3)['p']:.3f}")
draw(network, M7)

In [ ]:
# Lesson 7.2 — the chain rule for w11 (equation 7.3), factor by factor
f1 = v['p'] - 1                  # ① loss ← z: p̂ − y
f2 = network['v1']                # ② z ← a1: v1
f3 = v['a1'] * (1 - v['a1'])     # ③ a1 ← z1: derivative of the sigmoid, using the ACTIVATION a1
f4 = M7[0]                       # ④ z1 ← w11: x1
d_w11 = f1 * f2 * f3 * f4
print(f"① {f1:.3f}  ② {f2:g}  ③ {f3:.4f}  ④ {f4}  → derivative = {d_w11:.3f}")

# Check with a small numerical perturbation (like the secants of lesson 3.3)
def loss_w11(w):
    return loss(1, forward(dict(network, w11=w), M7)['p'])
h = 0.0001
print(f"numerical derivative: ({loss_w11(0.5 + h):.6f} − {loss_w11(0.5 - h):.6f}) / (2h) = "
      f"{(loss_w11(0.5 + h) - loss_w11(0.5 - h)) / (2 * h):.4f}")

# One update by hand, η = 0.5, then we redo the forward pass
eta = 0.5
w11_new = round(0.5 - eta * d_w11, 3)   # rounded to the thousandth, as in the book: 0.655
v_n = forward(dict(network, w11=w11_new), M7)
print(f"w11: 0.5 → {w11_new:.3f}; z1 = {v_n['z1']:.2f}, a1 = {v_n['a1']:.4f}, z = {v_n['z']:.3f}, "
      f"p̂ = {v_n['p']:.3f}, loss = {loss(1, v_n['p']):.3f}")
draw(network, M7, backward=True)

ws = np.linspace(-1, 3, 200)
plt.figure(figsize=(6, 3))
plt.plot(ws, [loss_w11(w) for w in ws], color='dimgrey', label='loss of M7 as a function of w11')
plt.plot(ws[25:126], loss_w11(0.5) + d_w11 * (ws[25:126] - 0.5), '--', color=BACKWARD, label='tangent at 0.5')
plt.scatter([0.5, w11_new], [loss_w11(0.5), loss_w11(w11_new)], color=['black', FORWARD], zorder=3)
plt.xlabel('w11'); plt.ylabel('loss of M7'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lesson 7.3 — backpropagation: the nine derivatives (equation 7.4), then an update of ALL the parameters
def backward(P, x, y):
    v = forward(P, x)
    d = v['p'] - y                                   # error signal of the output
    d1 = d * P['v1'] * v['a1'] * (1 - v['a1'])       # error signal of N1
    d2 = d * P['v2'] * v['a2'] * (1 - v['a2'])       # error signal of N2
    return {'w11': d1 * x[0], 'w12': d1 * x[1], 'b1': d1, 'w21': d2 * x[0], 'w22': d2 * x[1], 'b2': d2,
            'v1': d * v['a1'], 'v2': d * v['a2'], 'c': d}


def update(P, g, eta):
    return {k: P[k] - eta * g[k] for k in P}        # all derivatives are taken BEFORE changing anything


g = backward(network, M7, 1)
new = update(network, g, 0.5)
print(pd.DataFrame({'value': network, 'derivative': g, 'new value': new}).round(3))
print(f"M7: p̂ = {forward(new, M7)['p']:.3f}, loss = {loss(1, forward(new, M7)['p']):.3f}")
print(f"M3 before: p̂ = {forward(network, M3)['p']:.3f}, loss = {loss(0, forward(network, M3)['p']):.3f}; "
      f"after: p̂ = {forward(new, M3)['p']:.3f}, loss = {loss(0, forward(new, M3)['p']):.3f}")

## Lesson 7.4 — Learning in small batches: mini-batches and epochs

The network in its starting state (D7.1), the twelve messages of chapter 5. To follow the calculation by hand, only the output constant $c$ moves. Derivative of a batch = mean of the derivatives of its messages (equation 7.5).

In [ ]:
# Lesson 7.4 — sum s, probability and error signal δ = p̂ − y of the twelve messages, at the start (c = 0)
X12 = list(zip(twelve['links'], twelve['suspicious_words']))
y12 = [1 if e == 'spam' else 0 for e in twelve['label']]


def p_with_c(x, c_out):
    return forward(dict(network, c=c_out), x)['p']    # only the output constant changes


deltas = np.array([p_with_c(x, 0) - y_ for x, y_ in zip(X12, y12)])
print(pd.DataFrame({'message': twelve['message'], 'y': y12, 's': [forward(network, x)['z'] for x in X12],
                    'p̂': [p_with_c(x, 0) for x in X12], 'δ': deltas}).round(3).to_string(index=False))
print(f"derivative with respect to c — M7 alone: {deltas[6]:.3f}; batch M1 to M4: {deltas[:4].mean():.3f}; "
      f"twelve: sum {deltas.sum():.3f}, mean {deltas.mean():.3f}")
print(f"batches M5 to M8: {deltas[4:8].mean():.3f}; M9 to M12: {deltas[8:].mean():.3f}; "
      f"mean of the three batches: {np.mean([deltas[:4].mean(), deltas[4:8].mean(), deltas[8:].mean()]):.3f}")


def loss_c(c_out):
    return np.mean([loss(y_, p_with_c(x, c_out)) for x, y_ in zip(X12, y12)])


# One epoch: three updates, each on its own batch, each starting from the value left by the previous one
c_batch, path = 0.0, [0.0]
for k in range(3):
    batch = range(4 * k, 4 * k + 4)
    d = np.mean([p_with_c(X12[i], c_batch) - y12[i] for i in batch])
    c_batch = c_batch - 0.5 * d
    path.append(c_batch)
    print(f"update {k + 1} (M{4 * k + 1} to M{4 * k + 4}): derivative {d:.4f} → c = {c_batch:.3f}")
c_one_step = 0 - 0.5 * deltas.mean()
print(f"mean loss of the twelve: {loss_c(0):.3f} → {loss_c(c_batch):.3f} after one epoch (3 updates); "
      f"a single step on the twelve: c = {c_one_step:.3f}, loss {loss_c(c_one_step):.3f}")
plt.figure(figsize=(6, 2.6))
plt.plot(range(4), path, marker='o', color=FORWARD, label='one epoch, batches of four')
plt.plot([0, 1], [0, c_one_step], marker='s', color='black', label='one step on the twelve')
plt.plot([0, 1], [0, -0.5 * deltas[6]], marker='^', color=BACKWARD, linestyle='--', label='M7 alone')
plt.xlabel('update'); plt.ylabel('constant c'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Short questions (lesson 7.4)
updates_per_epoch = 12 // 3                      # n / m = 4
updates_ten_epochs = 10 * updates_per_epoch         # 40
c_batch_M6_M7 = 0 - 0.5 * deltas[5:7].mean()    # derivative ½ (0.500 − 0.330) ≈ 0.085, so c ≈ −0.042

In [ ]:
# Check of the short questions
if updates_per_epoch is None or updates_ten_epochs is None or c_batch_M6_M7 is None:
    print('not yet completed. Complete the three lines.')
else:
    print(f'1. {updates_per_epoch} per epoch, {updates_ten_epochs} for ten epochs →',
          'correct' if (updates_per_epoch, updates_ten_epochs) == (4, 40) else 'check again')
    print(f'2. c = {c_batch_M6_M7:.3f} →', 'correct' if abs(c_batch_M6_M7 + 0.5 * deltas[5:7].mean()) < 1e-9 else 'check again')

## Optional workshop — training the network on the twelve messages

1,000 steps of gradient descent ($\eta = 0.5$), starting from the weights of lesson 7.1, averaging at each step the derivatives of the twelve messages of chapter 5. **All the measurements below are on these twelve messages, which are used for training**: they say nothing about how the network behaves on new messages (lesson 6.3).

In [ ]:
# Optional workshop: training on the twelve messages (measured on the training data)
X12 = list(zip(twelve['links'], twelve['suspicious_words']))
y12 = [1 if e == 'spam' else 0 for e in twelve['label']]


def loss_12(P):
    return np.mean([loss(y, forward(P, x)['p']) for x, y in zip(X12, y12)])


def gradient_12(P):
    gs = [backward(P, x, y) for x, y in zip(X12, y12)]
    return {k: np.mean([g[k] for g in gs]) for k in P}     # derivative of the mean loss = mean of the derivatives


P_trained, history = dict(network), [loss_12(network)]
for _ in range(1000):
    P_trained = update(P_trained, gradient_12(P_trained), 0.5)
    history.append(loss_12(P_trained))
print(f"mean loss on the twelve training messages: before {history[0]:.3f}, after {history[-1]:.3f}")
for name, x in (('M7', M7), ('M3', M3)):
    print(f"{name}: p̂ before {forward(network, x)['p']:.3f}, after {forward(P_trained, x)['p']:.3f}")
print('errors at the 0.5 threshold on the twelve: before', sum((forward(network, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)),
      ', after', sum((forward(P_trained, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)))
print('weights after training:', {k: round(float(v), 2) for k, v in P_trained.items()})
plt.figure(figsize=(6, 3)); plt.plot(history, color=FORWARD)
plt.xlabel('step'); plt.ylabel("mean loss (twelve training messages)"); plt.grid(alpha=0.4); plt.show()

## Exercise 7.1 — Reading a colleague's network

*Concepts: layers, weights and biases (lesson 7.1, equations 7.1 and 7.2).* Data D7.E1: the dictionary `colleague`.

### a) to e) Annotated solution

**a)** Six weights, in the hidden layer ($w_{11}$, $w_{12}$, $w_{21}$, $w_{22}$) and at the output ($v_1$, $v_2$), and three biases ($b_1$, $b_2$, $c$): nine parameters, like the network of lesson 7.1.

**b)** $w_{21} = -1$ connects the links to neuron N2; $w_{12} = 1$ connects the suspicious words to neuron N1.

**c)** The bias −4 belongs to N1. The output bias is $c = -2$.

**d)** With $x_1 = x_2 = 0$, the products with the weights are zero: $z_1 = b_1 = -4$ and $z_2 = b_2 = -1$. The bias is the weighted sum of an "empty" message.

**e)** Here, N2 computes $-x_1 + 2x_2 - 1$: it grows with the suspicious words and decreases with the links, and its output weight $v_2 = 1$ is positive: when its activation increases, the probability of spam **goes up**. In the network of lesson 7.1, N2 computed $x_1 - x_2$ and its output weight was negative: almost the opposite effect.

In [ ]:
n_parameters = len(colleague)
empty = forward(colleague, (0, 0))

In [ ]:
# Check of a) and d)
if n_parameters is None or empty is None:
    print('not yet completed. Complete the two lines.')
else:
    print(f"{n_parameters} parameters; empty message: z1 = {empty['z1']:g}, z2 = {empty['z2']:g} →",
          'correct' if n_parameters == 9 and empty['z1'] == -4 and empty['z2'] == -1 else 'check again')

## Exercise 7.2 — Computing an output by hand

*Concepts: forward pass (equations 7.1 and 7.2), sigmoid (equation 5.4), loss (equation 5.6), decision rule (equation 5.1).*

### a) to c) Annotated solution

**a)** M6: $z_1 = 1 \times 2 + 1 \times 2 - 4 = 0$, $a_1 = \sigma(0) = 0.5$; $z_2 = -1 \times 2 + 2 \times 2 - 1 = 1$, $a_2 = \sigma(1) \approx 0.7311$; $z = 2 \times 0.5 + 1 \times 0.7311 - 2 \approx -0.269$; $\hat p = \sigma(-0.269) \approx 0.433$.

**b)** $0.433 < 0.5$: M6 is not flagged, which is right since it is legitimate. Its loss is $-\ln(1 - 0.433) \approx 0.568$.

**c)** M7: $z_1 = 2 + 3 - 4 = 1$, $a_1 \approx 0.7311$; $z_2 = -2 + 6 - 1 = 3$, $a_2 \approx 0.9526$; $z = 1.4621 + 0.9526 - 2 \approx 0.415$; $\hat p \approx 0.602$, loss $-\ln(0.602) \approx 0.507$. The network of lesson 7.1 gives M7 a smaller loss, 0.401: it gives it a probability closer to 1.

In [ ]:
p_M6 = forward(colleague, (2, 2))['p']
loss_M6 = loss(0, p_M6)                          # M6 is legitimate: y = 0
loss_M7 = loss(1, forward(colleague, M7)['p'])      # M7 is spam: y = 1

In [ ]:
# Check of a) to c)
if p_M6 is None or loss_M6 is None or loss_M7 is None:
    print('not yet completed. Complete the three lines.')
else:
    print(f"M6: p̂ = {p_M6:.3f}, flagged: {p_M6 >= 0.5}, loss = {loss_M6:.3f} →",
          'correct' if abs(p_M6 - 0.4332) < 5e-4 and abs(loss_M6 - 0.5677) < 5e-4 else 'check again')
    print(f"M7: loss {loss_M7:.3f} (colleague) versus {loss(1, forward(network, M7)['p']):.3f} (lesson 7.1) →",
          'correct' if abs(loss_M7 - 0.5071) < 5e-4 else 'check again')
    draw(colleague, (2, 2))

## Exercise 7.3 — Interpreting an activation

*Concepts: role of a hidden neuron (lesson 7.1).* Network of lesson 7.1, twelve messages of chapter 5.

### a) to d) Annotated solution

**a)** $a_2$ exceeds 0.8 for M2 (0.881) and M3 (0.982): two legitimate messages, with links and **no** suspicious words. $a_2$ is exactly 0.5 when $z_2 = x_1 - x_2 = 0$, that is, when the message has as many links as suspicious words: M4, M6, M9.

**b)** N2 lights up (activation above 0.5) when a message has more links than suspicious words, and all the more so as the gap grows. Its output weight $v_2 = -2$ then lowers the output score, and so the probability of spam. Among these six messages, the two where N2 lights up the most are legitimate; but six messages, with hand-picked weights, are not enough to say that N2 "spots newsletters": it would have to be checked on many new messages.

**c)** M6: $z = 2 \times 0.5 - 2 \times 0.5 + 0 = 0$ exactly, so $\hat p = 0.5$ exactly. With the "greater than or equal" rule, M6 is flagged: a false alarm, right at the threshold. This is the situation of M3 in chapter 5, whose raw score was exactly 0.

**d)** It is N2: with $a_2 \approx 0.982$, it removes about 1.96 from the output score, and the network gives M3 only 0.276. An activation is not a probability of spam: it is a number between 0 and 1 that serves as an ingredient for the output. Only $\hat p$, at the output, is an estimated probability.

In [ ]:
twelve['a2'] = [forward(network, (l, m))['a2'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
lit_up = list(twelve.loc[twelve['a2'] > 0.8, 'message'])

In [ ]:
# Check of a) and c)
if lit_up is None or twelve['a2'].isna().any():
    print('not yet completed. Complete the two lines.')
else:
    twelve['a1'] = [forward(network, (l, m))['a1'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
    twelve['p̂'] = [forward(network, (l, m))['p'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
    print(twelve.round(3).to_string(index=False))
    print('a2 > 0.8:', lit_up, '→', 'correct' if lit_up == ['M2', 'M3'] else 'check again')
    print(f"M6: z = {forward(network, (2, 2))['z']}, p̂ = {forward(network, (2, 2))['p']} → flagged at the 0.5 threshold: {forward(network, (2, 2))['p'] >= 0.5}")

## Exercise 7.4 — Completing a derivative in Jupyter

*Concepts: chain rule (equation 7.3), numerical check (lesson 7.2).* The weight $w_{22}$ connects the suspicious words to neuron N2.

### a) and b) Annotated solution

**a)** ① $\hat p - y \approx -0.3303$; ② $v_2 = -2$; ③ $a_2(1 - a_2) = 0.2689 \times 0.7311 \approx 0.1966$; ④ $x_2 = 3$. Product: $-0.3303 \times (-2) \times 0.1966 \times 3 \approx 0.390$.

**b)** The sign is positive: increasing $w_{22}$ makes the loss **go up**. Follow the signs: an increase in $w_{22}$ raises $z_2$ (because $x_2 = 3 > 0$), and so $a_2$; since $v_2 < 0$, $z$ goes down; $\hat p$ goes down, whereas M7 is a spam whose probability is already too low. Two negative factors, ① and ②, give a positive product.

In [ ]:
def derivative_w22(x, y):
    v = forward(network, x)
    return (v['p'] - y) * network['v2'] * v['a2'] * (1 - v['a2']) * x[1]   # ① × ② × ③ × ④

In [ ]:
# Check of c): chain rule versus numerical perturbation
if derivative_w22(M7, 1) is None:
    print('not yet completed. Complete the function.')
else:
    h = 0.0001
    num = (loss(1, forward(dict(network, w22=-1 + h), M7)['p']) - loss(1, forward(dict(network, w22=-1 - h), M7)['p'])) / (2 * h)
    print(f"chain rule: {derivative_w22(M7, 1):.4f} →", 'correct' if abs(derivative_w22(M7, 1) - 0.3896) < 5e-4 else 'check again')
    print(f"numerical derivative: {num:.4f} → gap {abs(num - derivative_w22(M7, 1)):.1e} →",
          'correct' if abs(num - derivative_w22(M7, 1)) < 1e-6 else 'check again')

## Exercise 7.5 — Predict, then check: the effect of a change in weight

*Concepts: derivative and approximation (lessons 3.3 and 7.2).*

### a) Annotated solution

With the derivative alone: loss $\approx 0.401 - 0.310 \times \Delta w$. For $\Delta w = 0.1$: $0.401 - 0.031 = 0.370$. For $\Delta w = 2$: $0.401 - 0.620 \approx -0.22$, a **negative** loss, which is impossible.

In [ ]:
def loss_after(dw):
    return loss(1, forward(dict(network, w11=0.5 + dw), M7)['p'])   # redo the whole forward pass

In [ ]:
# Check of b)
if loss_after(0.1) is None:
    print('not yet completed. Complete the function.')
else:
    L0 = loss(1, forward(network, M7)['p'])
    for dw in (0.1, 2):
        print(f"dw = {dw}: predicted {L0 + d_w11 * dw:.3f}, exact {loss_after(dw):.3f}")
    print('→', 'correct' if abs(loss_after(0.1) - 0.3716) < 5e-4 and abs(loss_after(2) - 0.2126) < 5e-4 else 'check again')

### b) and c) Annotated solution

Exact losses: 0.372 for $w_{11} = 0.6$, very close to the predicted 0.370; 0.213 for $w_{11} = 2.5$, very far from the predicted −0.22. The derivative describes the curve **near** the starting point: the tangent hugs the curve for a small step, then moves away from it (figure in lesson 7.2). For a large step, the loss flattens out and cannot become negative. This is why the learning rate stays small: each step must stay in the zone where the derivative tells the truth, even if that means taking many steps.

## Exercise 7.6 — Challenge: updating several weights on two messages

*Concepts: backpropagation (equation 7.4), mean loss (equation 5.7), gradient descent (equation 3.7).* Messages M7 (spam) and M3 (legitimate).

### a) to c) Annotated solution

**a)** Starting losses: 0.401 for M7 and $-\ln(1 - 0.2761) \approx 0.323$ for M3; mean $\approx 0.362$.

**b)** $\delta_{M7} = 0.6697 - 1 \approx -0.330$ and $\delta_{M3} = 0.2761 - 0 = 0.2761$. Derivatives of the mean loss: $\partial L/\partial v_1 = (-0.330 \times 0.6225 + 0.2761 \times 0.5)/2 \approx -0.034$; $\partial L/\partial v_2 = (-0.330 \times 0.2689 + 0.2761 \times 0.9820)/2 \approx 0.091$; $\partial L/\partial c = (-0.330 + 0.2761)/2 \approx -0.027$.

**c)** With $\eta = 1$: $v_1 \approx 2.034$, $v_2 \approx -2.091$, $c \approx 0.027$. The hidden activations do not change. New losses: about 0.393 for M7 and 0.311 for M3, mean 0.352. The mean loss goes down, and so does **each** of the two: unlike the update on M7 alone (lesson 7.3), M3 is not sacrificed, because its derivative enters the mean.

In [ ]:
def mean_gradient(P):
    g7, g3 = backward(P, M7, 1), backward(P, M3, 0)
    return {k: (g7[k] + g3[k]) / 2 for k in P}    # derivative of a mean = mean of the derivatives

In [ ]:
# Check of c), then d): the nine parameters, one step then 100 steps
def mean_loss(P):
    return (loss(1, forward(P, M7)['p']) + loss(0, forward(P, M3)['p'])) / 2

if mean_gradient(network) is None:
    print('not yet completed. Complete the function.')
else:
    g = mean_gradient(network)
    output_only = dict(network, v1=network['v1'] - g['v1'], v2=network['v2'] - g['v2'], c=network['c'] - g['c'])
    print(f"start: mean loss {mean_loss(network):.3f}")
    print(f"c) v1, v2, c only: M7 {loss(1, forward(output_only, M7)['p']):.3f}, "
          f"M3 {loss(0, forward(output_only, M3)['p']):.3f}, mean {mean_loss(output_only):.3f} →",
          'correct' if abs(mean_loss(output_only) - 0.3519) < 5e-4 else 'check again')
    nine = update(network, g, 1)
    print(f"d) the nine parameters, one step: mean {mean_loss(nine):.3f} →",
          'correct' if abs(mean_loss(nine) - 0.2584) < 5e-4 else 'check again')
    P, history = dict(network), []
    for _ in range(100):
        P = update(P, mean_gradient(P), 1)
        history.append(mean_loss(P))
    print(f"   100 steps: p̂(M7) = {forward(P, M7)['p']:.3f}, p̂(M3) = {forward(P, M3)['p']:.3f}, mean loss {history[-1]:.3f}")
    plt.figure(figsize=(6, 3)); plt.plot(range(1, 101), history, color=FORWARD)
    plt.xlabel('step'); plt.ylabel('mean loss (M7 and M3)'); plt.grid(alpha=0.4); plt.show()

### d) Annotated solution

One step on the nine parameters brings the mean loss down from 0.362 to 0.258: much more than the output layer alone (0.352), because the hidden neurons adapt too. After 100 steps, the network gives 0.982 to M7 and 0.018 to M3: mean loss 0.018. This success proves nothing for what comes next: with nine parameters for two messages, the network can learn them by heart, like filter B in lesson 6.3. Only messages it has not seen would tell what it is worth.